In [1]:
import pandas as pd
import numpy as np

In [2]:
demo = pd.read_sas("../data/raw/P_DEMO.xpt", format='xport', encoding='infer')
ghb = pd.read_sas("../data/raw/P_GHB.xpt", format='xport', encoding='infer')

assert demo['SEQN'].is_unique
assert ghb['SEQN'].is_unique

# print(len(demo['SEQN']), len(ghb['SEQN']))

ghb = ghb.rename(columns={'SEQN': 'id'}).astype({'id': int})

ghb

,id,LBXGH
0,109264,5.3
1,109266,5.2
2,109271,5.6
3,109273,5.1
4,109274,5.7
...,...,...
10404,124815,5.8
10405,124817,6.6
10406,124818,5.9
10407,124821,5.9


In [3]:
demo_engineered = demo.copy()

maps = {
    'RIDSTATR': {1: False, 2: True},
    'RIDRETH3': {
        1: 'mexican american',
        2: 'other hispanic',
        3: 'non-hispanic white',
        4: 'non-hispanic black',
        6: 'non-hispanic asian',
        7: 'other/multi-racial'
    },
    'RIDEXMON': {1: 'november_to_april', 2: 'may_to_october'},
    'DMDBORN4': {1: 'usa_born', 2: 'non_usa_born'},
    'DMDYRUSZ': {1: '< 5', 2: '5<=x<15', 3: '15<=x<30', 4: '>=30'},
    'RIAGENDR': {1: 'male', 2: 'female'},
    'SIALANG': {1: 'english', 2: 'spanish'},
    'DMDEDUC2': {
        1: 'less_than_9th_grade',
        2: 'some_high_school',
        3: 'high_school_grad',
        4: 'some_college',
        5: 'college_grad'
    },
    'DMDMARTZ': {
        1: 'married_with_partner',
        2: 'widowed_divorced_separated',
        3: 'never_married'
    },
    'SIAINTRP': {1: True, 2: False},
    'RIDEXPRG': {1: True, 2: False}
}

# Refused / don't know / cannot be determined -> become NaN
missing_codes = {
    'DMDBORN4': {77, 99},
    'DMDYRUSZ': {77, 99},
    'DMDEDUC2': {7, 9},
    'DMDMARTZ': {77, 99},
    'RIDEXPRG': {3}
}

for raw_col, map_dict in maps.items():
    raw_codes = set(demo[raw_col].dropna().unique())
    allowed = set(map_dict) | missing_codes.get(raw_col, set())
    assert raw_codes <= allowed, f"{raw_col}: unexpected codes {raw_codes - allowed}"

    is_bool = all(isinstance(v, bool) for v in map_dict.values())
    demo_engineered[raw_col] = demo[raw_col].map(map_dict).astype('boolean' if is_bool else 'category')
demo_engineered.loc[demo_engineered['WTMECPRP'] < 1e-10, 'WTMECPRP'] = 0

demo_engineered['SEQN'] = demo_engineered['SEQN'].astype('int')
demo_engineered = demo_engineered.rename(columns={
    'SEQN' : 'id',
    'RIAGENDR' : 'sex',
    'RIDSTATR' : 'is_examined',
    'RIDAGEYR' : 'age_at_screening',
    'RIDRETH3' : 'race',
    'RIDEXMON' : 'examination_period',
    'DMDBORN4' : 'birth_place',
    'DMDYRUSZ' : 'years_in_us',
    'SIALANG' : 'language_of_interview',
    'DMDEDUC2' : 'level_of_education',
    'DMDMARTZ' : 'marital_status',
    'SIAINTRP' : 'interpreter_used',
    'RIDEXPRG' : 'is_pregnant',
    'INDFMPIR' : 'income_to_poverty_ratio',
    'WTMECPRP' : 'exam_weight'
}, errors='raise')


In [4]:
demo_engineered = demo_engineered.drop(
    ['SDDSRVYR', 'RIDRETH1', 'FIALANG','FIAPROXY', 
     'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 
     'AIALANGA', 'RIDAGEMN', 'WTINTPRP', 'SIAPROXY'],
    axis=1
)


In [5]:
merged = demo_engineered.merge(
    ghb,
    how='inner',
    on='id',
    validate='one_to_one'
)

merged = merged.rename(columns={
    'LBXGH' : 'hba1c',
    "SDMVPSU" : "survey_psu",
    "SDMVSTRA" : "survey_strata"
}, errors='raise')

assert merged['is_examined'].all()

adults = merged[merged['hba1c'].notna()]
print('with HbA1c:', len(adults))

adults = adults[adults['age_at_screening'] >= 20]
print('aged 20+:', len(adults))

adults = adults[~adults['is_pregnant'].fillna(False)]
print('not pregnant:', len(adults))

display(adults)

with HbA1c: 9737
aged 20+: 8081
not pregnant: 8005


,id,is_examined,sex,age_at_screening,race,examination_period,birth_place,years_in_us,level_of_education,marital_status,is_pregnant,language_of_interview,interpreter_used,exam_weight,survey_psu,survey_strata,income_to_poverty_ratio,hba1c
1,109266,True,female,29.0,non-hispanic asian,may_to_october,non_usa_born,5<=x<15,college_grad,never_married,False,english,False,8154.968193,2.0,168.0,5.00,5.2
2,109271,True,male,49.0,non-hispanic white,may_to_october,usa_born,NaN,some_high_school,never_married,<NA>,english,False,8658.732873,1.0,167.0,NaN,5.6
3,109273,True,male,36.0,non-hispanic white,may_to_october,usa_born,NaN,some_college,never_married,<NA>,english,False,22163.596850,1.0,155.0,0.83,5.1
4,109274,True,male,68.0,other/multi-racial,november_to_april,usa_born,NaN,some_college,never_married,<NA>,english,False,7801.600218,2.0,167.0,1.20,5.7
7,109282,True,male,76.0,non-hispanic white,may_to_october,usa_born,NaN,college_grad,married_with_partner,<NA>,english,False,31816.346500,2.0,164.0,3.61,5.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10404,124815,True,male,52.0,non-hispanic black,may_to_october,usa_born,NaN,some_college,widowed_divorced_separated,<NA>,english,False,27768.450036,1.0,172.0,5.00,5.8
10405,124817,True,female,67.0,mexican american,november_to_april,non_usa_born,>=30,some_high_school,married_with_partner,<NA>,spanish,False,8189.572200,2.0,165.0,2.19,6.6
10406,124818,True,male,40.0,non-hispanic black,november_to_april,usa_born,NaN,college_grad,married_with_partner,<NA>,english,False,21666.889837,1.0,166.0,3.82,5.9
10407,124821,True,male,63.0,non-hispanic black,november_to_april,usa_born,NaN,some_high_school,widowed_divorced_separated,<NA>,english,False,4853.430230,1.0,158.0,3.71,5.9


In [6]:
diq = pd.read_sas('../data/raw/P_DIQ.xpt', format='xport', encoding='infer')
relevant_cols = ['SEQN', 'DIQ010', 'DIQ050', 'DIQ070']

diq_engineered = diq[relevant_cols].copy()


diq_engineered['SEQN'] = diq_engineered['SEQN'].astype('int')

diq_maps = {
    'DIQ010': {1 : 'yes', 2 : 'no', 3: 'borderline'},
    'DIQ050': {1 : True, 2: False},
    'DIQ070': {1: True, 2: False}
}

diq_missing_codes = {
    'DIQ010': {7, 9},
    'DIQ050': {7, 9},
    'DIQ070': {7, 9},
}

for raw_col, map_dict in diq_maps.items():
    raw_codes = set(diq[raw_col].dropna().unique())
    allowed = set(map_dict) | diq_missing_codes.get(raw_col, set())
    assert raw_codes <= allowed, f"{raw_col}: unexpected codes {raw_codes - allowed}"
    
    is_bool = all(isinstance(v, bool) for v in map_dict.values())
    diq_engineered[raw_col] = diq[raw_col].map(map_dict).astype('boolean' if is_bool else 'category')


diq_engineered = diq_engineered.rename(columns={
    'SEQN' : 'id',
    'DIQ010': 'told_diabetes',
    'DIQ050' : 'takes_insulin',
    'DIQ070' : 'takes_diabetes_pills'
},errors='raise')

adults_diq = adults.merge(
    diq_engineered,
    how='left',
    on='id',
    validate='one_to_one'
)

adults_diq.insert(1,'is_diagnosed', 
                  (adults_diq['told_diabetes'] == 'yes') 
                  | adults_diq['takes_insulin'].fillna(False)
                  | adults_diq['takes_diabetes_pills'].fillna(False)
                  )

assert adults_diq['is_diagnosed'].notna().all()

adults_diq = adults_diq.loc[adults_diq['told_diabetes'].notna()]
print('known diabetes status:', len(adults_diq))

adults_diq = adults_diq.loc[~adults_diq['is_diagnosed']]
print("not diagnosed: ", len(adults_diq))

adults_diq['undiagnosed_diabetes'] = (adults_diq['hba1c'] >= 6.5).astype('int')
display(adults_diq['undiagnosed_diabetes'].value_counts(), adults_diq['undiagnosed_diabetes'].value_counts(normalize=True))


known diabetes status: 8000
not diagnosed:  6644


undiagnosed_diabetes
0    6407
1     237
Name: count, dtype: int64

undiagnosed_diabetes
0    0.964329
1    0.035671
Name: proportion, dtype: float64

In [7]:
assert adults_diq['is_examined'].all()
assert (~adults_diq['is_pregnant'].fillna(False)).all()
analysis_df = adults_diq.drop(['hba1c', "told_diabetes", 
                              "takes_insulin", "takes_diabetes_pills",
                              "is_diagnosed", "is_pregnant", "is_examined"], axis=1)
n_before = len(analysis_df)


In [8]:
bmx = pd.read_sas("../data/raw/P_BMX.xpt", format='xport', encoding='infer')

bmx_engineered = bmx[['SEQN' ,'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST']].copy()

bmx_engineered = bmx_engineered.rename(columns={
    'SEQN' : 'id',
    'BMXWT' : 'weight', 
    'BMXHT' : 'height',
    'BMXBMI' : 'bmi',
    'BMXWAIST' : 'waist_circ'
}, errors='raise')

bmx_engineered['id'] = bmx_engineered['id'].astype('int')

analysis_df = analysis_df.merge(
    bmx_engineered,
    how='left',
    on='id',
    validate='one_to_one'
)

assert len(analysis_df) == n_before

In [9]:
display(analysis_df[['weight', 'height', 'bmi', 'waist_circ']].isna().sum())

weight         78
height         89
bmi            94
waist_circ    275
dtype: int64

In [10]:
bpxo = pd.read_sas("../data/raw/P_BPXO.xpt", format="xport", encoding='infer')

bpxo = bpxo.rename(columns={
    'SEQN': 'id',

},errors='raise')

bpxo['systolic_bp'] = bpxo[['BPXOSY1', 'BPXOSY2', 'BPXOSY3']].mean(axis=1)
bpxo['diastolic_bp'] = bpxo[['BPXODI1', 'BPXODI2', 'BPXODI3']].mean(axis=1)

bpxo['id'] = bpxo['id'].astype('int')

bpxo_engineered = bpxo[['id', 'systolic_bp', 'diastolic_bp']]

analysis_df = analysis_df.merge(
    bpxo_engineered,
    how='left',
    on='id',
    validate='one_to_one'
)

print(analysis_df[['systolic_bp', 'diastolic_bp']].describe())

assert len(analysis_df) == n_before

       systolic_bp  diastolic_bp
count  6006.000000   6006.000000
mean    123.761849     74.971723
std      18.646422     11.479268
min      76.333333     41.333333
25%     110.333333     67.000000
50%     121.000000     74.000000
75%     133.666667     82.000000
max     218.666667    135.000000


In [11]:
display(analysis_df[['systolic_bp' , 'diastolic_bp']].isna().sum())

systolic_bp     638
diastolic_bp    638
dtype: int64

In [12]:
bpq = pd.read_sas('../data/raw/P_BPQ.xpt', format='xport', encoding='infer')



yes_no = {1: True, 2: False}

bpq_maps = {col: yes_no for col in ['BPQ020', 'BPQ050A', 'BPQ080', 'BPQ100D']}
bpq_missing_codes = {col: {7, 9} for col in bpq_maps}

bpq_engineered = bpq[['SEQN', *bpq_maps.keys()]].copy()

for raw_col, map_dict in bpq_maps.items():
    raw_codes = set(bpq[raw_col].dropna().unique())
    allowed = set(map_dict) | bpq_missing_codes.get(raw_col, set())
    assert raw_codes <= allowed, f"{raw_col}: unexpected codes {raw_codes - allowed}"
    
    is_bool = all(isinstance(v, bool) for v in map_dict.values())
    bpq_engineered[raw_col] = bpq[raw_col].map(map_dict).astype('boolean' if is_bool else 'category')

bpq_engineered = bpq_engineered.rename(columns={
    'SEQN' : 'id',
    'BPQ020' : 'told_high_bp',
    'BPQ080' : 'told_high_chol',
    'BPQ050A': 'takes_bp_meds', 
    'BPQ100D': 'takes_chol_meds' 
}, errors='raise')
bpq_engineered['id'] = bpq_engineered['id'].astype('int')


analysis_df = analysis_df.merge(
    bpq_engineered,
    how='left',
    on='id',
    validate='one_to_one'
)

analysis_df['takes_bp_meds'] = analysis_df['takes_bp_meds'].fillna(False)
analysis_df['takes_chol_meds'] = analysis_df['takes_chol_meds'].fillna(False)

assert len(analysis_df) == n_before

In [13]:
display(analysis_df)

,id,sex,age_at_screening,race,examination_period,birth_place,years_in_us,level_of_education,marital_status,language_of_interview,interpreter_used,exam_weight,survey_psu,survey_strata,income_to_poverty_ratio,undiagnosed_diabetes,weight,height,bmi,waist_circ,systolic_bp,diastolic_bp,told_high_bp,takes_bp_meds,told_high_chol,takes_chol_meds
0,109266,female,29.0,non-hispanic asian,may_to_october,non_usa_born,5<=x<15,college_grad,never_married,english,False,8154.968193,2.0,168.0,5.00,0,97.1,160.2,37.8,117.9,99.000000,54.333333,False,False,True,False
1,109271,male,49.0,non-hispanic white,may_to_october,usa_born,NaN,some_high_school,never_married,english,False,8658.732873,1.0,167.0,NaN,0,98.8,182.3,29.7,120.4,107.000000,67.000000,False,False,True,True
2,109273,male,36.0,non-hispanic white,may_to_october,usa_born,NaN,some_college,never_married,english,False,22163.596850,1.0,155.0,0.83,0,74.3,184.2,21.9,86.8,113.666667,67.333333,False,False,False,False
3,109282,male,76.0,non-hispanic white,may_to_october,usa_born,NaN,college_grad,married_with_partner,english,False,31816.346500,2.0,164.0,3.61,0,83.3,177.1,26.6,NaN,139.333333,72.666667,True,True,True,False
4,109284,female,44.0,mexican american,november_to_april,non_usa_born,15<=x<30,some_high_school,married_with_partner,spanish,False,13794.793542,1.0,154.0,NaN,0,91.1,152.7,39.1,103.1,NaN,NaN,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6639,124814,male,64.0,non-hispanic black,may_to_october,usa_born,NaN,high_school_grad,widowed_divorced_separated,english,False,3866.882086,1.0,171.0,2.00,0,114.3,174.5,37.5,119.5,138.666667,86.666667,False,False,False,False
6640,124815,male,52.0,non-hispanic black,may_to_october,usa_born,NaN,some_college,widowed_divorced_separated,english,False,27768.450036,1.0,172.0,5.00,0,94.3,178.8,29.5,99.3,140.000000,86.666667,False,False,False,False
6641,124818,male,40.0,non-hispanic black,november_to_april,usa_born,NaN,college_grad,married_with_partner,english,False,21666.889837,1.0,166.0,3.82,0,108.8,168.7,38.2,114.7,103.000000,67.333333,False,False,False,False
6642,124821,male,63.0,non-hispanic black,november_to_april,usa_born,NaN,some_high_school,widowed_divorced_separated,english,False,4853.430230,1.0,158.0,3.71,0,79.5,176.4,25.5,97.1,124.000000,66.666667,True,True,True,False


In [14]:
display(analysis_df[['told_high_bp','told_high_chol','takes_bp_meds', 'takes_chol_meds']].isna().sum())

told_high_bp        6
told_high_chol     33
takes_bp_meds       0
takes_chol_meds     0
dtype: int64

In [15]:
tchol = pd.read_sas('../data/raw/P_TCHOL.xpt', format='xport', encoding='infer')
hdl = pd.read_sas('../data/raw/P_HDL.xpt', format='xport', encoding='infer')

tchol_engineered = tchol[['SEQN', 'LBXTC']].copy()
hdl_engineered = hdl[['SEQN', 'LBDHDD']].copy()

tchol_engineered = tchol_engineered.rename(columns={
    'SEQN' : 'id',
    'LBXTC' : 'total_chol'
},errors='raise')

hdl_engineered = hdl_engineered.rename(columns={
    'SEQN' : 'id',
    'LBDHDD' : 'hdl_chol'
},errors='raise')

for data in [tchol_engineered, hdl_engineered]: 
    data['id'] = data['id'].astype('int')
    analysis_df = analysis_df.merge(
        data,
        how='left',
        on='id',
        validate='one_to_one'
    )
    assert len(analysis_df) == n_before

In [16]:
display(analysis_df[['total_chol','hdl_chol']].isna().sum())
display(analysis_df[['total_chol','hdl_chol']].describe())

total_chol    138
hdl_chol      138
dtype: int64

,total_chol,hdl_chol
count,6506.000000,6506.000000
mean,188.064556,54.467722
std,39.578682,16.132092
min,79.000000,10.000000
25%,160.000000,43.000000
50%,185.000000,52.000000
75%,212.750000,63.000000
max,428.000000,189.000000


In [ ]:
smoking = pd.read_sas('../data/raw/P_SMQ.xpt', format='xport', encoding='infer')

smoking_engineered = smoking[['SEQN', 'SMQ020','SMQ040']].copy()

smq_allowed = {
    'SMQ020': {1, 2, 7, 9},
    'SMQ040': {1, 2, 3, 7, 9}
}

for raw_col, allowed in smq_allowed.items():
    raw_codes = set(smoking_engineered[raw_col].dropna().unique())
    assert raw_codes <= allowed, f"{raw_col}: unexpected codes {raw_codes - allowed}"

ever_smoked = smoking_engineered['SMQ020']
smokes_now = smoking_engineered['SMQ040']


conditions = [
    ever_smoked == 2,
    (ever_smoked == 1) & (smokes_now == 3),
    (ever_smoked == 1) & smokes_now.isin([1, 2])
]

choices = ['never', 'former', 'current']

smoking_engineered['smoking_status'] = pd.Series(
    np.select(conditions, choices, default=None),
    index=smoking_engineered.index
).astype('category')

smoking_engineered = smoking_engineered.rename(columns={'SEQN': 'id'}, errors='raise')
smoking_engineered['id'] = smoking_engineered['id'].astype('int')
smoking_engineered = smoking_engineered[['id', 'smoking_status']]


analysis_df = analysis_df.merge(
    smoking_engineered,
    how='left',
    on='id',
    validate='one_to_one'
)

assert len(analysis_df) == n_before

display(analysis_df['smoking_status'].value_counts(dropna=False))

smoking_status
never      3931
former     1466
current    1245
NaN           2
Name: count, dtype: int64